In [1]:
import os
from PIL import Image
from tqdm import tqdm
from io import BytesIO
import numpy as np

import common

2024-03-21 18:06:14.402776: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-03-21 18:06:14.402798: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-03-21 18:06:14.403452: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-03-21 18:06:14.407352: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-03-21 18:06:15.065445: W tensorflow/compiler/tf2

In [2]:
DATASET_PTH = 'theodore_yolo'

In [3]:
# read names
with open("data/train.txt", "rt") as f:
    train_names = [line.strip() for line in f.readlines()]

with open("data/val.txt", "rt") as f:
    val_names = [line.strip() for line in f.readlines()]

with open("data/test.txt", "rt") as f:
    test_names = [line.strip() for line in f.readlines()]


In [4]:
os.makedirs(DATASET_PTH, exist_ok=True)
os.makedirs(os.path.join(DATASET_PTH, "train"), exist_ok=True)
os.makedirs(os.path.join(DATASET_PTH, "val"), exist_ok=True)
os.makedirs(os.path.join(DATASET_PTH, "test"), exist_ok=True)

In [5]:
ds = common.load_dataset("data/training_25k_4.tfrecord", common.parse_feature)
for example in tqdm(ds):
    height = int(example['image/height'].values)
    width = int(example['image/width'].values)

    img_string = (example['image/encoded'].numpy())
    img_filename = (example['image/filename'].numpy()).decode('utf-8')
    
    if img_filename in train_names:
        subset_name = "train"
    elif img_filename in val_names:
        subset_name = "val"
    else:
        print(f"WARNING: img {img_filename} has not subset.")
        continue

    pil_img = Image.open(BytesIO(img_string))

    bb_xmin = np.array(example['image/object/bbox/xmin'].values)
    bb_ymin = np.array(example['image/object/bbox/ymin'].values)
    bb_xmax = np.array(example['image/object/bbox/xmax'].values)
    bb_ymax = np.array(example['image/object/bbox/ymax'].values)
    labels = np.array(example['image/object/class/label'].values)

    yolo_file_lines = []
    for xmin, ymin, xmax, ymax, label in zip(bb_xmin, bb_ymin, bb_xmax, bb_ymax, labels):
        w = xmax - xmin
        h = ymax - ymin
        yolo_file_lines.append(f"{label - 1} {xmin + w / 2} {ymin + h / 2} {w} {h}\n")

    pil_img.save(os.path.join(DATASET_PTH, subset_name, img_filename))
    with open(os.path.join(DATASET_PTH, subset_name, img_filename.replace(".jpg", ".txt")), "wt") as f:
        f.writelines(yolo_file_lines)

2024-03-21 18:07:01.219379: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-03-21 18:07:01.238488: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-03-21 18:07:01.238624: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [6]:

ds = common.load_dataset("data/training_25k_1.tfrecord", common.parse_feature)
subset_name = "test"
for example in tqdm(ds):
    height = int(example['image/height'].values)
    width = int(example['image/width'].values)

    img_string = (example['image/encoded'].numpy())
    img_filename = (example['image/filename'].numpy()).decode('utf-8')
    
    if img_filename not in test_names:
        continue

    pil_img = Image.open(BytesIO(img_string))

    bb_xmin = np.array(example['image/object/bbox/xmin'].values)
    bb_ymin = np.array(example['image/object/bbox/ymin'].values)
    bb_xmax = np.array(example['image/object/bbox/xmax'].values)
    bb_ymax = np.array(example['image/object/bbox/ymax'].values)
    labels = np.array(example['image/object/class/label'].values)

    yolo_file_lines = []
    for xmin, ymin, xmax, ymax, label in zip(bb_xmin, bb_ymin, bb_xmax, bb_ymax, labels):
        w = xmax - xmin
        h = ymax - ymin
        yolo_file_lines.append(f"{label - 1} {xmin + w / 2} {ymin + h / 2} {w} {h}\n")

    pil_img.save(os.path.join(DATASET_PTH, subset_name, img_filename))
    with open(os.path.join(DATASET_PTH, subset_name, img_filename.replace(".jpg", ".txt")), "wt") as f:
        f.writelines(yolo_file_lines)

0it [00:00, ?it/s]

25000it [01:10, 356.98it/s]


In [7]:
len(os.listdir(os.path.join(DATASET_PTH, "train"))), len(os.listdir(os.path.join(DATASET_PTH, "val"))), len(os.listdir(os.path.join(DATASET_PTH, "test")))

(45000, 5002, 5000)

In [8]:
def read_pbtxt_to_dict(pbtxt_file):
    data_dict = {}
    with open(pbtxt_file, 'r') as file:
        item_dict = {}
        for line in file:
            line = line.strip()
            if line.startswith("item {"):
                item_dict = {}
            elif line.startswith("}"):
                data_dict[item_dict['id']] = item_dict['display_name']
            else:
                key, value = line.split(':', 1)
                item_dict[key.strip()] = value.strip().strip('"')
    return data_dict

# Replace 'your_pbtxt_file_path.pbtxt' with the path to your pbtxt file
pbtxt_file_path = 'data/label_map.pbtxt'
labels = read_pbtxt_to_dict(pbtxt_file_path)
print(labels)

{'1': 'person', '2': 'chair', '3': 'table', '4': 'armchair', '5': 'tv', '6': 'wheeled walker'}


In [9]:
with open(os.path.join(DATASET_PTH, "project.yaml"), "w") as f:
    f.write("path: " + os.path.abspath(DATASET_PTH) + "\n")
    f.write("train: train\n")
    f.write("val: val\n")
    f.write("test: test\n")
    f.write("\n")
    f.write("# class names\n")
    f.write("names:\n")
    for class_idx, class_name in labels.items():
        f.write(f"  {int(class_idx) - 1}: '{class_name}'\n")